<h1 align="center">LAB1_S3. Static word-embeddings for text representation</h1>

<h3 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h3>
<h3 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h3>
<h3 style="display:block; margin-top:5px;" align="center">2024-2025</h3>
<h3 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h3>
<br>

### Put your names here

- Bartosz Stoklosa
- Baranyi Sándor

In [1]:
#Installing Gensim library
!pip install -U gensim
!pip install -U nltk
!pip install -U fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=4313472 sha256=b3eff0216a9e5b172511008c89636d7b77ff31d5c7795d7e079ac4310e803273
  Stored in directory: /root/.cache/pip/wheels/65/4f/35/5057db0249224e9ab55a513fa6b79451473ceb7713017823c3
Successfully built fasttext


## Some libraries

In [1]:
#import fasttext.util
import gensim.downloader as api
from gensim.models.keyedvectors import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt_tab")
nltk.download("punkt")
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Sanyi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sanyi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sanyi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Sanyi\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sanyi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load both corpora

In [2]:
path_english = "EXIST2024_EN_examples.csv"
path_spanish = "EXIST2024_ES_examples.csv"

df = {
    "english": pd.read_csv(path_english, sep="\t"),
    "spanish": pd.read_csv(path_spanish, sep="\t")
}

# OR USING Google colab
#from google.colab import drive
#drive.mount('/content/drive')
#df = {
#    "english": pd.read_csv("/content/drive/MyDrive/LNR/LNR2025/Lab1/EXIST2024_EN_examples.csv", sep="\t"),
#    "spanish": pd.read_csv("/content/drive/MyDrive/LNR/LNR2025/Lab1/EXIST2024_ES_examples.csv", sep="\t")
#}


## Preprocess and tokenize the corpora:

- remove URLs
- remove hashtags
- remove users
- lowercase
- tokenize (using _word_tokenize_)
- remove stopwords (using nltk stopwords)

Note: tokenization and stopword removal are language-dependent.

In [3]:
# Info: nltk.tokenize.word_tokenize(text, language='english', preserve_line=False)

web_re = re.compile(r"https?:\/\/[^\s]+", re.U)
user_re = re.compile(r"(@\w+\-?(?:\w+)?)", re.U)
hashtag_re = re.compile(r"(#\w+\-?(?:\w+)?)", re.U)

stopw = {
    "english": nltk.corpus.stopwords.words("english"),
    "spanish": nltk.corpus.stopwords.words("spanish")
}

def preprocess(text):
    text = re.sub(web_re, '', text)
    text = re.sub(user_re, '', text)
    text = re.sub(hashtag_re, '', text)
    return text.lower()

def tokenize(text_list, lang="english"):

    token_list = [nltk.tokenize.word_tokenize(preprocess(line), language=lang, preserve_line=False) for line in text_list]
    token_list = [[word for word in line if word not in stopw[lang]] for line in token_list]
    return token_list

tokenized_text = {
    "english": tokenize(df["english"]["text"], "english"),
    "spanish": tokenize(df["spanish"]["text"], "spanish")
}

t = ["Hola, mi nombre es Antonio, ¿todo bien? https://www.upv.es @paquita", "Hi! my name is Peter"]
print(t)
print(tokenize(t, "spanish"))
print(tokenize(t, "english"))

['Hola, mi nombre es Antonio, ¿todo bien? https://www.upv.es @paquita', 'Hi! my name is Peter']
[['hola', ',', 'nombre', 'antonio', ',', '¿todo', 'bien', '?'], ['hi', '!', 'my', 'name', 'is', 'peter']]
[['hola', ',', 'mi', 'nombre', 'es', 'antonio', ',', '¿todo', 'bien', '?'], ['hi', '!', 'name', 'peter']]


## Text representation using static embeddings

ENGLISH

- word2vec-google-news-300 (using Gemini)
- fasttext-wiki-news-subwords-300 (using Gemini)
- glove-wiki-gigaword-300 (using Gemini)

SPANISH
- Fasttext (https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz) (using Gemini)

### Load the models

In [6]:
import gensim.downloader as api
english = api.load("word2vec-google-news-300")
english2 = api.load("fasttext-wiki-news-subwords-300")
english3 = api.load("glove-wiki-gigaword-300")


In [5]:
#!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz
#Loading the pre-trained word embedding vectors from a text file format
spanish = KeyedVectors.load_word2vec_format('cc.es.300.vec.gz', binary=False, limit=None)

### Compute static word-embeddings representation of the tweets

In [7]:
def gensim_sentence_rep(tokens, model):
    """
    Compute the word-embedding representation of a list of tokens by averaging
        the representations of each word
    """
    dim = model.vector_size
    zero_vec=np.zeros(dim)
    avg_vec=np.zeros(dim)
    total_w=0
    for w in tokens:
        try:
            avg_vec+=model[w]
            total_w+=1
        except:
           #print(f"The word {w} is not found in the embedding matrix")
            continue
    if total_w == 0:
        return zero_vec

    return avg_vec / total_w

# Apply "gensim_sentence_rep"
# COMPLETE
english_text = tokenized_text["english"]
spanish_text = tokenized_text["spanish"]
english_avg_1 = [gensim_sentence_rep(tokens, english) for tokens in english_text]
english_avg_2 = [gensim_sentence_rep(tokens, english2) for tokens in english_text]
english_avg_3 = [gensim_sentence_rep(tokens, english3) for tokens in english_text]
spanish_avg = [gensim_sentence_rep(tokens, spanish) for tokens in spanish_text]


## Compute cosine similarities

In [8]:

# COMPLETE
def compute_cosine_similarity(matrix):
    res = cosine_similarity(matrix, matrix)
    sim= np.round(cosine_similarity(res,res), 3)
    return sim

cos_sim_eng_1 = compute_cosine_similarity(english_avg_1)
cos_sim_eng_2 = compute_cosine_similarity(english_avg_2)
cos_sim_eng_3 = compute_cosine_similarity(english_avg_3)
cos_sim_spanish = compute_cosine_similarity(spanish_avg)


## Show results

In [20]:
# COMPLETE

def show_results(matrix):
    no_idx_1 = 4
    no_idx_2 = 5
    yes_idx_1 = 0
    yes_idx_2 = 1
    print("label: YES")
    print(f"sentence1: {df['english'].iloc[yes_idx_1]['text']}\n-----------------\nsentence2: {df['english'].iloc[yes_idx_2]['text']}")
    print(f"distance: {matrix[yes_idx_1][yes_idx_2]}")
    print(f"\n\n")
    print("label: NO")
    print(f"sentence1: {df['english'].iloc[no_idx_1]['text']}\n-----------------\nsentence2: {df['english'].iloc[no_idx_2]['text']}")
    print(f"distance: {matrix[no_idx_1][no_idx_2]}")

def show_results_es(matrix):
    no_idx_1 = 1
    no_idx_2 = 2
    yes_idx_1 = 0
    yes_idx_2 = 3
    print("label: YES")
    print(f"sentence1: {df['spanish'].iloc[yes_idx_1]['text']}\n-----------------\nsentence2: {df['spanish'].iloc[yes_idx_2]['text']}")
    print(f"distance: {matrix[yes_idx_1][yes_idx_2]}")
    print(f"\n\n")
    print("label: NO")
    print(f"sentence1: {df['spanish'].iloc[no_idx_1]['text']}\n-----------------\nsentence2: {df['spanish'].iloc[no_idx_2]['text']}")
    print(f"distance: {matrix[no_idx_1][no_idx_2]}")

print("w2v300\n======\n\n")
show_results(cos_sim_eng_1)
print("\n\n\nftsub300\n======\n\n")
show_results(cos_sim_eng_2)
print("\n\n\nglwiki300\n======\n\n")
show_results(cos_sim_eng_3)
print("\n\n\nftes300\n======\n\n")
show_results_es(cos_sim_spanish)


w2v300


label: YES
sentence1: Writing a uni essay in my local pub with a coffee. Random old man keeps asking me drunk questions when I'm trying to concentrate &amp; ends with "good luck, but you'll just end up getting married and not use it anyway". #EverydaySexism is alive and well 🙃
-----------------
sentence2: @UniversalORL it is 2021 not 1921. I dont appreciate that on two rides by myself your team member looked behind me and asked the man behind how many in my party. Not impressed #everydaysexism
distance: 0.994



label: NO
sentence1: New to the shelves this week - looking forward to reading these books @malorieblackman @jenlynnbarnes  @EverydaySexism #readingforpleasure #newbooks https://t.co/Q3IfmlSBCB
-----------------
sentence2: I guess that’s fairly normal for a Neanderthal https://t.co/UkFl8tNH4F
distance: 0.967



ftsub300


label: YES
sentence1: Writing a uni essay in my local pub with a coffee. Random old man keeps asking me drunk questions when I'm trying to concentrat